# The Price is Right

## 第 8 周日程安排

第 1 天：Modal.com 与 SpecialistAgent  
第 2 天：RAG、FrontierAgent、Ensemble Agent  
第 3 天：ScannerAgent、MessengerAgent   
第 4 天：AutonomousPlannerAgent  
第 5 天：The Price Is Right 终章


现在轮到 Planning Agent 了

In [ ]:
# 导入：OpenAI Tools（函数调用）、Scanner、Chroma
# 今天演示 Autonomous Planning：模型自己决定调用哪些工具

import json
from openai import OpenAI
from dotenv import load_dotenv
from agents.scanner_agent import ScannerAgent
import chromadb
import logging
load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-5.1"

## 先用一些测试数据开始

In [ ]:
# 用测试数据模拟「扫描全网优惠」的返回结果

test_results = ScannerAgent().test_scan()
test_results

## 现在我们来创建 3 个假装的函数..

In [ ]:
# 假工具 1：扫描优惠（先返回硬编码结果，方便调试 Tool Calling）

def scan_the_internet_for_bargains() -> str:
    """ This tool scans the internet for great deals and gets a curated list of promising deals """
    print("Fake function to scan the internet - this returns a hardcoded set of deals")
    return test_results.model_dump_json()

In [ ]:
# 假工具 2：估值（固定返回 $300，先跑通工具循环）

def estimate_true_value(description: str) -> str:
    """
    This tool estimates the true value of a product based on a text description of it
    """
    print(f"Fake function to estimating true value of {description[:20]}... - this always returns $300")
    return f"Product {description} has an estimated true value of $300"

In [ ]:
# 假工具 3：通知用户发现超值优惠

def notify_user_of_deal(description: str, deal_price: float, estimated_true_value: float, url: str) -> str:
    """
    This tool notifies the user of a great deal, given a description of it, the price of the deal, and the estimated true value
    """
    print(f"Fake function to notify user of {description} which costs {deal_price} and estimate is {estimated_true_value}")
    return "notification sent ok"

### 我们来试用一下

In [ ]:
# 先手动调用一次通知工具，确认函数本身可用

notify_user_of_deal("a new iphone", 100, 1000, "https://www.apple.com/iphone")

### 好，现在是一大段 JSON

In [ ]:
# 下面是 OpenAI Tools 的 JSON Schema：告诉模型每个函数的名字、参数与用途
# 模型不会执行代码，只会按 schema「申请」调用；真正执行在 handle_tool_call

scan_function = {
        "name": "scan_the_internet_for_bargains",
        "description": "Returns top bargains scraped from the internet along with the price each item is being offered for",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False
        }
    }

estimate_function = {
    "name": "estimate_true_value",
    "description": "Given the description of an item, estimate how much it is actually worth",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item to be estimated"
            },
        },
        "required": ["description"],
        "additionalProperties": False
    }
}

notify_function = {
    "name": "notify_user_of_deal",
    "description": "Send the user a push notification about the single most compelling deal; only call this one time",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "string",
                "description": "The description of the item itself scraped from the internet"
            },
            "deal_price": {
                "type": "number",
                "description": "The price offered by this deal scraped from the internet"
            }
            ,
            "estimated_true_value": {
                "type": "number",
                "description": "The estimated actual value that this is worth"
            }
            ,
            "url": {
                "type": "string",
                "description": "The URL of this deal as scraped from the internet"
            }
        },
        "required": ["description", "deal_price", "estimated_true_value", "url"],
        "additionalProperties": False
    }
}

In [ ]:
# 把三个函数 schema 打包成 tools 列表，传给 chat.completions

tools = [{"type": "function", "function": scan_function},
 {"type": "function", "function": estimate_function},
 {"type": "function", "function": notify_function}
 ]

In [ ]:
# 查看 tools 结构

tools

In [ ]:
# 根据模型返回的 tool_calls，用 globals() 找到同名 Python 函数并执行
# 再把结果以 role=tool 的消息喂回对话

def handle_tool_call(message):
    """
    Actually call the tools associated with this message
    """
    results = []
    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
# 系统与用户指令：要求模型按「扫描 → 估值 → 通知最优」的流程使用工具
# 注意：不要改动下面的英文 prompt 字符串

system_message = "You find great deals on bargain products using your tools, and notify the user of the best bargain."
user_message = """
First, use your tool to scan the internet for bargain deals. Then for each deal, use your tool to estimate its true value.
Then pick the single most compelling deal where the price is much lower than the estimated true value, and use your tool to notify the user.
Then just reply OK to indicate success.
"""
messages = [{"role": "system", "content": system_message},{"role": "user", "content": user_message}]

In [ ]:
# 查看初始 messages

messages

In [ ]:
# Tool-calling 循环：若 finish_reason 是 tool_calls 就执行工具并继续对话，直到模型说完

done = False
while not done:
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        results = handle_tool_call(message)
        messages.append(message)
        messages.extend(results)
    else:
        done = True
response.choices[0].message.content

## 接下来……进入 Autonomous Planning Agent

并把假函数替换成真正的函数！

In [ ]:
# 打开日志，观察 AutonomousPlanningAgent

root = logging.getLogger()
root.setLevel(logging.INFO)

In [ ]:
# 连接商品向量库，供规划 Agent 做 RAG 估值

DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collection = client.get_or_create_collection('products')

In [ ]:
# AutonomousPlanningAgent：用规划 + 工具调用自主完成「找优惠」任务

from agents.autonomous_planning_agent import AutonomousPlanningAgent
agent = AutonomousPlanningAgent(collection)

In [ ]:
# 启动自主规划：扫描、估值、挑选并通知

agent.plan()